# Topic: Statistics: Outlier Detection (Z-score vs IQR)

## Definition (30-second explanation)
* **Outliers** are data points that differ significantly from other observations in a dataset, potentially caused by measurement errors, natural variations, or fraudulent activity.
* **Z-score** identifies outliers by measuring how many standard deviations a data point is from the mean.
* **IQR Method (Tukey Fences)** identifies outliers using percentiles, specifically looking for values that fall outside 1.5 times the Interquartile Range below the 1st quartile or above the 3rd quartile.

## Why Interviewers Ask This
* **Data Cleaning Skills:** Real-world data is extremely messy; interviewers want to see your standard operating procedure for exploratory data analysis (EDA).
* **Statistical Intuition:** To test if you know *which* method to apply based on the underlying distribution (normal vs. skewed).
* **Business Judgment:** To ensure you don't blindly delete valid extreme data points (like whale customers or legitimate large transactions) just because a formula flagged them.

## Core Concepts
* **Z-score Threshold:** Assuming a normal distribution, a standard threshold is $\vert{}z\vert{} > 3$, capturing data outside 99.7% of the expected range.
* **Interquartile Range (IQR):** The difference between the 75th percentile ($Q_3$) and 25th percentile ($Q_1$), representing the middle 50% of the data.
* **Tukey Fences:** The mathematical boundaries for the IQR method: Lower Bound = $Q_1 - 1.5 \times IQR$, Upper Bound = $Q_3 + 1.5 \times IQR$.
* **Model Sensitivity:** Linear regression and k-means clustering are highly sensitive to outliers. Tree-based models (Random Forest, XGBoost) are inherently robust to them.

## When to Use
* **Z-score:** Use when the data is roughly symmetric and normally distributed.
* **IQR Method:** Use when the data is highly skewed, non-normal, or already contains massive outliers (because the median/percentiles are robust to extreme values).
* **Advanced Methods:** Use Isolation Forests or DBSCAN when dealing with high-dimensional data where outliers exist across multiple features simultaneously.

## Advantages
* **Z-score:** Mathematically simple, easy to compute at scale, and directly interpretable as probabilities under a normal curve.
* **IQR:** Highly robust. The mean and standard deviation (used in Z-scores) are pulled by outliers, meaning outliers can "hide" themselves in Z-score calculations. IQR prevents this.
* **Visual Pairing:** Both map perfectly to visualizations (Histograms for Z-score, Box plots for IQR) for easy stakeholder communication.

## Limitations
* **Z-score:** Fails on skewed data. If applied to income data, almost no one will be flagged on the lower end, while too many are flagged on the upper end.
* **IQR:** The $1.5 \times$ multiplier is somewhat arbitrary. In massive datasets (e.g., millions of rows), it might flag too many legitimate points as outliers.

## Common Comparisons
* **Z-score vs IQR:** Z-score assumes normality and uses mean/variance. IQR makes no distributional assumptions and uses median/percentiles.
* **Trimming vs Winsorizing:** When handling outliers, "trimming" deletes them. "Winsorizing" caps them at a specific percentile (e.g., changing all values above the 99th percentile to equal the 99th percentile value).

## Common Interview Traps
* **The "Drop Everything" Trap:** Blindly writing a function to drop all outliers. You must state: "I would investigate them first to see if they are legitimate business events".
* **Applying Z-score to Skewed Data:** Using a Z-score on things like prices, salaries, or session lengths without applying a log-transform first.

## Python / SQL Syntax
```python
import numpy as np
import pandas as pd
from scipy import stats

# Z-score method
df['z_score'] = np.abs(stats.zscore(df['value']))
outliers_z = df[df['z_score'] > 3]

# IQR method
Q1 = df['value'].quantile(0.25)
Q3 = df['value'].quantile(0.75)
IQR = Q3 - Q1
outliers_iqr = df[(df['value'] < (Q1 - 1.5 * IQR)) | (df['value'] > (Q3 + 1.5 * IQR))]
```

## Important Formula
* **Z-score:** $z = \frac{x - \mu}{\sigma}$
* **IQR Bounds:** $[Q_1 - 1.5(IQR), Q_3 + 1.5(IQR)]$

## 45-Second Interview Answer
"To detect outliers, I choose between the Z-score and IQR methods based on the data's distribution. If the data is normally distributed, I use a Z-score threshold of 3. However, real-world data like prices or engagement metrics is usually right-skewed, so I prefer the IQR method, which uses robust percentiles to set boundaries at $1.5 \times IQR$ beyond the first and third quartiles. Most importantly, I never blindly drop outliers—I investigate them with domain experts, because an extreme value might be a measurement error, or it might be our highest-paying enterprise customer."

## Example Questions:

### Q1: 
**Scenario:**
"You find that 2% of transactions in a payment dataset have amounts over $100,000. How would you decide if these are legitimate transactions or outliers? How would you handle them before building a fraud detection model?"

**Ideal Interview Answer:**
"First, I would investigate the data source and use domain knowledge to determine if these are legitimate (e.g., B2B corporate payments) or system errors. If they are legitimate, I would not remove them, as doing so deletes valid business context. Instead, I would leave them in the dataset and use robust, tree-based machine learning algorithms (like XGBoost or Random Forest) which are naturally resistant to extreme values, rather than linear models. If they are highly suspicious or confirmed errors, I would flag them for manual review or apply a capping technique like Winsorization before training."

**Common Mistakes Candidates Make:**
* Suggesting to immediately delete the 2% of data to "clean" the dataset.
* Suggesting Z-scores for payment data (payment data is heavily right-skewed, so Z-scores would be mathematically inappropriate).

**One Likely Interviewer Follow-up:**
"If you *had* to use a Logistic Regression model for this fraud detection task because of strict interpretability requirements, how would you preprocess those valid $100,000 transactions so they don't ruin the model weights?" 
*(Answer: I would apply a log transformation to the transaction amounts to compress the extreme right tail and make the distribution more normal).*

## Practice Questions:

### Q1:
**You are a Data Scientist cleaning a housing dataset before training a pricing model. You have a Pandas DataFrame df containing house prices.**

*Write a Python function remove_iqr_outliers(df, column_name) that takes a DataFrame and a target column, calculates the Tukey fences (1.5 * IQR), and returns a new DataFrame with the outliers removed.*

In [13]:
# Data:
import pandas as pd
import numpy as np

# Sample mock dataset of house prices
df = pd.DataFrame({
    'house_id': [1, 2, 3, 4, 5, 6],
    'price': [250000, 265000, 240000, 275000, 2000000, 255000] # 2,000,000 is a clear outlier
})

In [16]:
df

,house_id,price
0,1,250000
1,2,265000
2,3,240000
3,4,275000
4,5,2000000
5,6,255000


In [14]:
import pandas as pd

def remove_iqr_outliers(df, column):
    # Calculate quartiles and IQR
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    
    # Define bounds
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Filter the dataframe and return a copy to prevent SettingWithCopyWarning
    filtered_df = df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]
    
    return filtered_df.copy()

In [15]:
remove_iqr_outliers(df= df, column= 'price')

,house_id,price
0,1,250000
1,2,265000
2,3,240000
3,4,275000
5,6,255000


**Common Mistakes Candidates Make:**
* **Using `and` instead of `&`:** In standard Python, `and` evaluates truthiness for scalar values. In Pandas, you must use the bitwise operator `&` for element-wise array comparisons.
* **Forgetting Parentheses:** Pandas requires each boolean condition to be wrapped in parentheses when combining them `(condition 1) & (condition 2)`, otherwise it throws a `ValueError` due to operator precedence.
* **Returning a View:** Forgetting `.copy()`, which leads to warnings if the filtered dataset is mutated later in the pipeline.

### Q2:
**You are building a data preprocessing pipeline for a financial machine learning model. You are given a DataFrame df with a highly volatile revenue column.**

**Write a Python script that creates three separate processed versions of this revenue data to test in your model pipeline:**
- Method 1 (Z-Score): Create df_zscore by removing any rows where the absolute Z-score of revenue is greater than 3.
- Method 2 (IQR): Create df_iqr by removing any rows where revenue falls outside the $1.5 \times IQR$ Tukey fences.
- Method 3 (Winsorizing/Capping): Create df_capped by capping the extreme revenue values at the 5th and 95th percentiles (do not drop any rows, just cap the values).

In [72]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.DataFrame({
    'transaction_id': [1, 2, 3, 4, 5, 6, 7],
    'revenue': [100, 105, 95, 110, 10000, 102, -500] 
})

In [73]:
df

,transaction_id,revenue
0,1,100
1,2,105
2,3,95
3,4,110
4,5,10000
5,6,102
6,7,-500


In [74]:
import pandas as pd
import numpy as np
from scipy.stats import zscore

# --- Method 1: Z-Score Removal ---
# Keeps only rows where the absolute Z-score is <= 3
df_zscore = df[np.abs(zscore(df['revenue'])) <= 3].copy()

# --- Method 2: IQR (Tukey Fences) Removal ---
Q1 = df['revenue'].quantile(0.25)
Q3 = df['revenue'].quantile(0.75)
IQR = Q3 - Q1
df_iqr = df[(df['revenue'] >= Q1 - 1.5 * IQR) & (df['revenue'] <= Q3 + 1.5 * IQR)].copy()

# --- Method 3: Winsorizing / Capping (Native Pandas) ---
# Better for production pipelines than dropping data entirely
lower_bound = df['revenue'].quantile(0.05)
upper_bound = df['revenue'].quantile(0.95)

df_capped = df.copy()
df_capped['revenue'] = df_capped['revenue'].clip(lower=lower_bound, upper=upper_bound)

In [75]:
df_zscore

,transaction_id,revenue
0,1,100
1,2,105
2,3,95
3,4,110
4,5,10000
5,6,102
6,7,-500


In [76]:
df_iqr

,transaction_id,revenue
0,1,100
1,2,105
2,3,95
3,4,110
5,6,102


In [77]:
df_capped

,transaction_id,revenue
0,1,100.0
1,2,105.0
2,3,95.0
3,4,110.0
4,5,7033.0
5,6,102.0
6,7,-321.5


**Common Mistakes Candidates Make:**
* **Confusing Trimming with Winsorizing:** Trimming deletes the rows; Winsorizing modifies the extreme values to equal a specified percentile threshold. 
* **Overcomplicating the Capping Logic:** Writing slow, custom `apply()` functions or `np.where()` chains to cap values instead of just using the highly optimized `df.clip()` method.
* **Mutating the Original Data:** Modifying the original dataframe without using `.copy()`, which destroys the raw data state and ruins the rest of the pipeline in a Jupyter Notebook environment.

**One Likely Interviewer Follow-up:**
"In a production machine learning pipeline, where would you calculate these $Q_1$, $Q_3$, or percentile bounds? Before or after doing your Train/Test split?" 
*(Answer: Strictly AFTER the train/test split, calculating the bounds only on the Training set and applying them to the Test set. Doing it before causes data leakage).*